# Au-dessus, en dessous : le pouls du cycle · *Above and below: the pulse of the cycle*

Notebook compagnon du chapitre **15. Croissance potentielle et output gap : la carte que personne ne peut mesurer** — [lire l'article](https://nmlab.io/ressources/croissance-potentielle-et-output-gap).
Companion notebook to chapter **15. Potential Growth and the Output Gap: The Map No One Can Measure** — [read the article](https://nmlab.io/en/ressources/potential-growth-and-output-gap).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure se régénère avec les **données FRED du jour**. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure with **today's FRED data**; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


from pandas import Series

def load_output_gap() -> Series:
    """Écart de production = (PIB réel − PIB potentiel) / PIB potentiel × 100.
    Output gap from real (GDPC1) and potential (GDPPOT) GDP, live from FRED."""
    real = nm.load_fred("GDPC1")
    potential = nm.load_fred("GDPPOT")
    return ((real / potential - 1) * 100).dropna()

gap = load_output_gap()
print(f"Dernier point / latest: {gap.index[-1]:%Y-%m} → {gap.iloc[-1]:+.1f} %")


import pandas as pd
import matplotlib.dates as mdates
from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Au-dessus, en dessous : le pouls du cycle",
        sub="Écart de production américain (PIB réel − potentiel), 1949-2026",
        ylab="écart de production, % du potentiel",
        today="aujourd'hui\n+1,0 %",
        gfc="Grande Récession\n−5,0 % (2009)",
        covid="COVID : −8,8 %\n(2020)",
        note="Positif (vert) = surchauffe ; négatif (rouge) = sous-emploi. ⚠ Ces valeurs dépendent du millésime CBO de\n"
             "février 2026 et seront révisées. Source : BEA et CBO via FRED."),
    "en": dict(
        title="Above and below: the pulse of the cycle",
        sub="U.S. output gap (real GDP − potential), 1949-2026",
        ylab="output gap, % of potential",
        today="today\n+1.0%",
        gfc="Great Recession\n−5.0% (2009)",
        covid="COVID: −8.8%\n(2020)",
        note="Positive (green) = overheating; negative (red) = slack. ⚠ These values depend on the February 2026 CBO\n"
             "vintage and will be revised. Source: BEA and CBO via FRED."),
}

def build_figure(gap: Series, lang: str) -> Figure:
    """Écart de production ombré : vert au-dessus de 0, rouge en dessous."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1045)
    ax = nm.axes(fig)
    ax.fill_between(gap.index, gap, 0, where=gap >= 0, color=nm.COLORS["green"],
                    alpha=0.58, interpolate=True)
    ax.fill_between(gap.index, gap, 0, where=gap < 0, color=nm.COLORS["rose"],
                    alpha=0.5, interpolate=True)
    ax.plot(gap.index, gap, color=nm.COLORS["text"], linewidth=1.5)
    ax.axhline(0, color=nm.COLORS["muted"], linewidth=1.4, alpha=0.9)
    ax.set_ylim(-10.2, 7.0)
    ax.set_yticks(range(-10, 7, 2))
    ax.set_ylabel(text["ylab"])
    ax.margins(x=0.01)
    ax.xaxis.set_major_locator(mdates.YearLocator(10))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.annotate(text["today"], xy=(gap.index[-1], float(gap.iloc[-1])),
                xytext=(pd.Timestamp("2019-06-01"), 5.2), ha="center", va="center",
                fontsize=22, fontweight="bold", color=nm.COLORS["green"], linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["green"], lw=1.8))
    ax.annotate(text["gfc"], xy=(pd.Timestamp("2009-09-01"), -4.9),
                xytext=(pd.Timestamp("2004-01-01"), -6.9), ha="center", va="center",
                fontsize=21, fontweight="bold", color=nm.COLORS["rose"], linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["rose"], lw=1.8))
    ax.annotate(text["covid"], xy=(pd.Timestamp("2020-04-01"), -8.6),
                xytext=(pd.Timestamp("2015-06-01"), -7.9), ha="center", va="center",
                fontsize=21, fontweight="bold", color=nm.COLORS["rose"], linespacing=1.5,
                arrowprops=dict(arrowstyle="->", color=nm.COLORS["rose"], lw=1.8))
    nm.header(fig, text["title"], text["sub"])
    nm.footer(fig, text["note"])
    return fig

build_figure(gap, LANG)